## Example 1: One ParsedStatement class parses a file line by line

In [ ]:
import importlib.metadata, os, tempfile
from dataclasses import dataclass
import flexparser as fp

print("flexparser", fp.__version__)
assert fp.__version__ == importlib.metadata.version("flexparser")


@dataclass(frozen=True)
class Assignment(fp.ParsedStatement):
    """`name <- value`"""
    lhs: str
    rhs: str

    @classmethod
    def from_string(cls, s):
        if "<-" not in s:
            return None  # not mine: let another statement class (or UnknownStatement) take it
        lhs, rhs = (part.strip() for part in s.split("<-"))
        return cls(lhs, rhs)


path = os.path.join(tempfile.mkdtemp(prefix="flexparser_nb_"), "source.txt")
with open(path, "w") as handle:
    handle.write("one <- other\nthree <- new value\none == three\n")
parsed = fp.parse(path, Assignment)
statements = list(parsed.iter_statements())
for statement in statements:
    print(" ", type(statement).__name__, statement.raw)
kinds = [type(statement).__name__ for statement in statements]
assert kinds == ["BOF", "Assignment", "Assignment", "UnknownStatement", "EOS"], kinds
assert (statements[2].lhs, statements[2].rhs) == ("three", "new value")
assert (statements[1].start_line, statements[3].start_line) == (1, 3)  # lines are numbered from 1
assert statements[3].raw == "one == three"
print("OK -- each line became a parsed statement; the line no class claimed is an UnknownStatement")

## Example 2: Two statement classes, and a ParsingError for a bad line

In [ ]:
import os, tempfile
from dataclasses import dataclass
import flexparser as fp


@dataclass
class InvalidIdentifier(fp.ParsingError):  # a ParsingError is an Exception, so not a frozen dataclass
    value: str


@dataclass(frozen=True)
class Assignment(fp.ParsedStatement):
    lhs: str
    rhs: str

    @classmethod
    def from_string(cls, s):
        if "<-" not in s:
            return None
        lhs, rhs = (part.strip() for part in s.split("<-"))
        if not lhs.isidentifier():
            return InvalidIdentifier(lhs)  # this is an assignment, but a wrong one
        return cls(lhs, rhs)


@dataclass(frozen=True)
class Equality(fp.ParsedStatement):
    lhs: str
    rhs: str

    @classmethod
    def from_string(cls, s):
        if "==" not in s:
            return None
        lhs, rhs = (part.strip() for part in s.split("=="))
        return cls(lhs, rhs)


path = os.path.join(tempfile.mkdtemp(prefix="flexparser_nb_"), "source.txt")
with open(path, "w") as handle:
    handle.write("one <- other\n2two <- new\none == three\n")
parsed = fp.parse(path, (Assignment, Equality))
body = [type(statement).__name__ for statement in parsed.iter_statements()][1:-1]  # without BOF and EOS
errors = list(parsed.errors())
print("statements:", body)
print("errors:", errors)
assert body == ["Assignment", "InvalidIdentifier", "Equality"], body
assert parsed.has_errors and len(errors) == 1 and errors[0].value == "2two", errors
assert errors[0].start_line == 2 and errors[0].raw == "2two <- new"
print("OK -- two statement classes share the file, and the bad assignment is reported as an error with its line")

## Example 3: A Block groups statements between begin and end

In [ ]:
import os, tempfile
from dataclasses import dataclass
import flexparser as fp


@dataclass(frozen=True)
class Assignment(fp.ParsedStatement):
    lhs: str
    rhs: str

    @classmethod
    def from_string(cls, s):
        if "<-" not in s:
            return None
        lhs, rhs = (part.strip() for part in s.split("<-"))
        return cls(lhs, rhs)


@dataclass(frozen=True)
class Begin(fp.ParsedStatement):
    @classmethod
    def from_string(cls, s):
        return cls() if s == "begin" else None


@dataclass(frozen=True)
class End(fp.ParsedStatement):
    @classmethod
    def from_string(cls, s):
        return cls() if s == "end" else None


class Config:
    pass


class AssignmentBlock(fp.Block[Begin, Assignment, End, Config]):
    """Assignments are only understood between `begin` and `end`."""


path = os.path.join(tempfile.mkdtemp(prefix="flexparser_nb_"), "source.txt")
with open(path, "w") as handle:
    handle.write("x <- 1\nbegin\na <- 2\nb <- 3\nend\n")
parsed = fp.parse(path, AssignmentBlock)
root = parsed[None].parsed_source  # None is the key of the file parse() was given
outside, block = root.body
print("outside the block:", type(outside).__name__, repr(outside.raw))
print("block:", type(block.opening).__name__, [(s.lhs, s.rhs) for s in block.body], type(block.closing).__name__)
assert isinstance(outside, fp.UnknownStatement)  # an assignment outside begin/end is not recognised
assert isinstance(block, AssignmentBlock)
assert isinstance(block.opening, Begin) and isinstance(block.closing, End)
assert [(s.lhs, s.rhs) for s in block.body] == [("a", "2"), ("b", "3")]
print("OK -- the Block groups begin / assignments / end, and assignments outside it are left unknown")

## Example 4: parse_bytes with custom delimiters, and the content hash

In [ ]:
from dataclasses import dataclass
import flexparser as fp


@dataclass(frozen=True)
class Setting(fp.ParsedStatement):
    key: str
    value: int

    @classmethod
    def from_string(cls, s):
        if "=" not in s:
            return None
        key, value = (part.strip() for part in s.split("=", 1))
        return cls(key, int(value))


content = b"width = 3 ; height = 4\ndepth=5"
# The delimiters decide where one statement ends: here ";" as well as the end of a line.
delimiters = {";": (fp.DelimiterInclude.SPLIT, fp.DelimiterAction.CONTINUE), **fp.SPLIT_EOL}
parsed = fp.parse_bytes(content, Setting, None, strip_spaces=True, delimiters=delimiters)
settings = {s.key: s.value for s in parsed.iter_statements() if isinstance(s, Setting)}
print("settings:", settings)
assert settings == {"width": 3, "height": 4, "depth": 5}, settings

# The same bytes give the same content hash, so a cached parse can be reused.
again = fp.parse_bytes(content, Setting, None, strip_spaces=True, delimiters=delimiters)
changed = fp.parse_bytes(content + b"\nx=1", Setting, None, strip_spaces=True, delimiters=delimiters)
first_hash = parsed[None].parsed_source.opening.content_hash
assert again[None].parsed_source.opening.content_hash == first_hash
assert changed[None].parsed_source.opening.content_hash != first_hash
print("content hash:", first_hash.algorithm_name, first_hash.hexdigest[:16] + "...")
print("OK -- parse_bytes split on ';' and newlines, and the content hash follows the bytes")

## Example 5: An include statement pulls in a second file

In [ ]:
import os, tempfile
from dataclasses import dataclass
import flexparser as fp


@dataclass(frozen=True)
class Include(fp.IncludeStatement):
    """`@include <file>`"""
    value: str

    @property
    def target(self):
        return self.value

    @classmethod
    def from_string(cls, s):
        if s.startswith("@include "):
            return cls(s[len("@include "):].strip())
        return None


@dataclass(frozen=True)
class Assignment(fp.ParsedStatement):
    lhs: str
    rhs: str

    @classmethod
    def from_string(cls, s):
        if "<-" not in s:
            return None
        lhs, rhs = (part.strip() for part in s.split("<-"))
        return cls(lhs, rhs)


folder = tempfile.mkdtemp(prefix="flexparser_nb_")
with open(os.path.join(folder, "main.txt"), "w") as handle:
    handle.write("a <- 1\n@include extra.txt\nc <- 3\n")
with open(os.path.join(folder, "extra.txt"), "w") as handle:
    handle.write("b <- 2\n")
parsed = fp.parse(os.path.join(folder, "main.txt"), (Include, Assignment))
print("sources parsed:", len(parsed))
assignments = [(s.lhs, s.rhs) for s in parsed.iter_statements() if isinstance(s, Assignment)]
print("assignments in reading order:", assignments)
assert len(parsed) == 2  # main.txt and the file it includes
assert assignments == [("a", "1"), ("b", "2"), ("c", "3")], assignments  # the include is read in place
assert not parsed.has_errors
print("OK -- @include pulled extra.txt in, and iter_statements() reads it where the include stands")